# 08 — SQL Integration within Jobs

> **DataMindAI | Ahmed**

## SQL tasks are first-class citizens in Lakeflow Jobs

| Direction | Mechanism |
|-----------|-----------|
| Parameters **into** SQL | `{{tasks.<task>.values.<key>}}` in the SQL file |
| SQL results **out** to Python | `{{tasks.<sql_task>.output.rows}}` as For-Each input |

## Scenario
1. Python task sets the `run_date` parameter
2. SQL task runs a filtered query using `{{tasks.set_params.values.run_date}}`
3. SQL output rows flow into a For-Each → one Python iteration per course

## SQL file setup
Save the SQL below as a `.sql` file in your Databricks Repo, then reference it
in the SQL task configuration.

---
## Step 1 — Python task sets parameters
---

In [ ]:
# ── TASK 1: set_params — publishes parameters for the SQL task ────────────────
import datetime

run_date    = '2024-01-15'
min_score   = 40
department  = 'all'

dbutils.jobs.taskValues.set(key='run_date',   value=run_date)
dbutils.jobs.taskValues.set(key='min_score',  value=int(min_score))
dbutils.jobs.taskValues.set(key='department', value=department)

print('Parameters published for SQL task:')
print(f'  run_date   = {run_date}')
print(f'  min_score  = {min_score}')
print(f'  department = {department}')

---
## Step 2 — SQL task (save as .sql file)
---

### course_summary.sql

Save this as `course_summary.sql` in your Databricks Repo.
In the Job UI SQL task, set the parameter:
```
run_date  =  {{tasks.set_params.values.run_date}}
min_score =  {{tasks.set_params.values.min_score}}
```

```sql
-- course_summary.sql
-- Parameters injected via {{tasks.set_params.values.X}}
-- Final SELECT becomes output rows: {{tasks.course_summary_sql.output.rows}}

SELECT
    course,
    COUNT(student_id)              AS total_students,
    SUM(CASE WHEN score >= :min_score THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN score < :min_score  THEN 1 ELSE 0 END) AS failed,
    ROUND(AVG(score), 1)           AS avg_score,
    ROUND(AVG(attendance), 1)      AS avg_attendance,
    SUM(CASE WHEN score < 40 OR attendance < 50 THEN 1 ELSE 0 END) AS at_risk
FROM demo_catalog.bronze.students_raw
GROUP BY course
ORDER BY avg_score DESC
```

In [ ]:
# ── Simulate what the SQL task returns ───────────────────────────────────────
# In a real job, you use a SQL task file. Here we run the SQL inline.

spark.table('demo_catalog.bronze.students_raw').createOrReplaceTempView('students')

result = spark.sql('''
    SELECT
        course,
        COUNT(student_id)                                        AS total_students,
        SUM(CASE WHEN score >= 40 THEN 1 ELSE 0 END)            AS passed,
        SUM(CASE WHEN score <  40 THEN 1 ELSE 0 END)            AS failed,
        ROUND(AVG(score), 1)                                     AS avg_score,
        ROUND(AVG(attendance), 1)                               AS avg_attendance,
        SUM(CASE WHEN score < 40 OR attendance < 50 THEN 1 ELSE 0 END) AS at_risk
    FROM students
    GROUP BY course
    ORDER BY avg_score DESC
''')

print('SQL task output rows (3 rows — one per course):')
result.show()

# Convert to list of dicts (what {{tasks.X.output.rows}} provides to downstream tasks)
output_rows = [r.asDict() for r in result.collect()]
import json
print('As Python list of dicts ({{tasks.sql_task.output.rows}} format):')
print(json.dumps(output_rows, indent=2))

---
## Step 3 — Python task consuming SQL output rows
---

### Job UI: For-Each consuming SQL output
```
For-Each task configuration:
  Inputs:      {{tasks.course_summary_sql.output.rows}}
  Concurrency: 3

Nested task parameters:
  course       = {{input.course}}
  total        = {{input.total_students}}
  avg_score    = {{input.avg_score}}
  at_risk      = {{input.at_risk}}
```

In [ ]:
# ── Consumer notebook: processes ONE course row from SQL output ───────────────
# In a job: receives {{input.X}} for each row from the SQL task
import json

try:
    raw = dbutils.widgets.get('input')
    row = json.loads(raw)
except:
    # Local fallback: first row from our SQL result
    row = {'course': 'Data Science', 'total_students': 4, 'passed': 3,
           'failed': 1, 'avg_score': 54.75, 'avg_attendance': 68.75, 'at_risk': 2}

course   = row['course']
total    = row['total_students']
passed   = row['passed']
avg_sc   = row['avg_score']
at_risk  = row['at_risk']
pass_rate = round((passed / total) * 100, 1) if total > 0 else 0

print(f'Processing SQL result for course: {course}')
print(f'  Total students : {total}')
print(f'  Passed         : {passed} ({pass_rate}%)')
print(f'  Avg score      : {avg_sc}')
print(f'  At-risk        : {at_risk}')

# Write course-level alert if at-risk threshold exceeded
if at_risk > 0:
    print(f'  ALERT: {at_risk} at-risk student(s) in {course}')
    from pyspark.sql import Row
    import datetime
    spark.createDataFrame([Row(
        course=course, at_risk_count=int(at_risk), avg_score=float(avg_sc),
        checked_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    )]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.course_alerts')

---
## Full SQL to Python pipeline — end to end
---

In [ ]:
# ── Simulate all three iterations in sequence ─────────────────────────────────
import json

# These are the rows the SQL task would return
sql_output_rows = [
    {'course': 'Mathematics',  'total_students': 3, 'passed': 3, 'failed': 0, 'avg_score': 72.3, 'at_risk': 0},
    {'course': 'Engineering',  'total_students': 3, 'passed': 3, 'failed': 0, 'avg_score': 68.3, 'at_risk': 0},
    {'course': 'Data Science', 'total_students': 4, 'passed': 3, 'failed': 1, 'avg_score': 54.8, 'at_risk': 2},
]

print('For-Each: iterating over SQL output rows')
print(f'Total iterations: {len(sql_output_rows)}')
print()

for i, row in enumerate(sql_output_rows, 1):
    pass_rate = round((row['passed']/row['total_students'])*100, 1)
    status = 'OK' if row['at_risk'] == 0 else f'ALERT: {row["at_risk"]} at-risk'
    print(f'[Iteration {i}] {row["course"]:15s} | Avg:{row["avg_score"]} | Pass:{pass_rate}% | {status}')